### Load Dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch

path = "raw/ArnavBajaj_34900787.npz"
data = np.load(path)
X = data["X"]
y = data["y"]

#free the file
data.close()

if path == "datasets/raw_data.npz":
    main_data_analysis = True
else:
    main_data_analysis = False

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

### Plot Indiviual Channels

In [ ]:
# concatenate windows along time
fs = 250
X_concat = np.concatenate(X, axis=1)

n_channels = X_concat.shape[0]
total_samples = X_concat.shape[1]

time = np.arange(total_samples) / fs

fig, axes = plt.subplots(n_channels, 1, figsize=(14, 12), sharex=True)

for ch in range(n_channels):
    axes[ch].plot(time, X_concat[ch])
    axes[ch].set_ylabel(f"Ch {ch+1}")
    axes[ch].grid(True)

axes[-1].set_xlabel("Time (seconds)")
fig.suptitle("Concatenated Raw EEG Across All Trials")
plt.tight_layout()
plt.show()

Calculate Quality Factors

In [ ]:
N, C, T = X.shape # n windows, n channels, n timepoints

rms = np.zeros((N, C))
peak = np.zeros((N, C))
spike = np.zeros(N)
corr_mean = np.zeros(N)

for i in range(N):
    w = X[i]

    rms[i] = np.sqrt(np.mean(w**2, axis=1))
    peak[i] = np.max(np.abs(w), axis=1)

    diff = np.diff(w, axis=1)
    spike[i] = np.max(np.abs(diff))

    corr = np.corrcoef(w)
    upper = corr[np.triu_indices_from(corr, k=1)]
    corr_mean[i] = np.mean(np.abs(upper))

rms_mu, rms_std = rms.mean(), rms.std()
peak_mu, peak_std = peak.mean(), peak.std()
spike_mu, spike_std = spike.mean(), spike.std()

bad_rms   = np.any(rms > rms_mu + 6 * rms_std, axis=1)
bad_peak  = np.any(peak > peak_mu + 10 * peak_std, axis=1)
bad_spike = spike > spike_mu + 10 * spike_std
bad_corr  = corr_mean > 0.95

bad = bad_rms | bad_peak | bad_spike | bad_corr
good = ~bad

X_clean = X[good]
y_clean = y[good]

print("Original windows:", len(X))
print("Kept windows:", len(X_clean))
print("Dropped:", bad.sum())

X_clean = np.clip(X_clean, -300, 300)  # µV

if main_data_analysis:
    np.savez(
        "datasets/clean_eye_data.npz",
        X=X_clean,
        y=y_clean
    )

EEG Visualization

In [ ]:
# concatenate windows along time
fs = 250
X_long = X_clean.transpose(1, 0, 2).reshape(C, -1)  # (C, N*T)
t_long = np.arange(X_long.shape[1]) / fs

fig, axes = plt.subplots(C, 1, figsize=(10, 2*C), sharex=True)

for ch in range(C):
    axes[ch].plot(t_long, X_long[ch])
    axes[ch].set_ylabel(f"Ch {ch}")
    axes[ch].grid(True)

axes[-1].set_xlabel("Time (s)")
fig.suptitle("EEG — concatenated cleaned windows")
plt.tight_layout()
plt.show()

Power Spectral Density (PSD)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch

fs = 250

# Frequency bands
bands = {
    "Delta (0.5–4)":  (0.5, 4),
    "Theta (4–8)":    (4, 8),
    "Alpha (8–13)":   (8, 13),
    "Beta (13–30)":   (13, 30),
    "Gamma (30–100)": (30, 100),
}

def bandpower_per_window(X, fs, bands):
    band_powers = {k: [] for k in bands}

    for w in X:  # windows
        ch_band = {k: [] for k in bands}

        for ch in range(w.shape[0]):
            f, Pxx = welch(w[ch], fs=fs, nperseg=fs//2)

            for band, (fmin, fmax) in bands.items():
                mask = (f >= fmin) & (f <= fmax)
                ch_band[band].append(np.trapezoid(Pxx[mask], f[mask]))

        # average channels
        for band in bands:
            band_powers[band].append(np.mean(ch_band[band]))

    # average windows
    return {band: np.mean(vals) for band, vals in band_powers.items()}


# Split by state
X_open   = X[y == 1]
X_closed = X[y == 2]

bp_open   = bandpower_per_window(X_open, fs, bands)
bp_closed = bandpower_per_window(X_closed, fs, bands)

# Plot
labels = list(bands.keys())
open_vals   = [bp_open[b] for b in labels]
closed_vals = [bp_closed[b] for b in labels]

x = np.arange(len(labels))
width = 0.35

plt.figure(figsize=(10, 4))
plt.bar(x - width/2, open_vals,   width, label="Eyes Open")
plt.bar(x + width/2, closed_vals, width, label="Eyes Closed")

plt.xticks(x, labels)
plt.ylabel("Average Band Power")
plt.title("EEG Band Power Comparison")
plt.legend()
plt.grid(axis="y")
plt.tight_layout()
plt.show()

In [ ]:
#Plot of Alpha waves of ch 0 and ch 1 as bar graphs inside same plot for open and closed eyes.
plt.figure(figsize=(12, 5))
alpha_open_ch0 = bandpower_per_window(X_open[:, 0:1], fs, {"Alpha": (8, 13)})["Alpha"]
alpha_closed_ch0 = bandpower_per_window(X_closed[:, 0:1], fs, {"Alpha": (8, 13)})["Alpha"]
alpha_open_ch1 = bandpower_per_window(X_open[:, 1:2], fs, {"Alpha": (8, 13)})["Alpha"]
alpha_closed_ch1 = bandpower_per_window(X_closed[:, 1:2], fs, {"Alpha": (8, 13)})["Alpha"]
x = np.arange(2)
width = 0.35
plt.bar(x - width/2, [alpha_open_ch0, alpha_open_ch1],   width, label="Eyes Open")
plt.bar(x + width/2, [alpha_closed_ch0, alpha_closed_ch1], width, label="Eyes Closed")
plt.xticks(x, ["Ch 0", "Ch 1"])
plt.ylabel("Average Alpha Power")
plt.title("Alpha Band Power by Channel")
plt.legend()
plt.grid(axis="y")
plt.tight_layout()
plt.show()

Feature Engineering

In [ ]:
import numpy as np
from scipy.signal import welch

def extract_band_features_per_channel(X, fs=250):
    """
    X: (N, C, T) EEG windows
    Returns:
        features: (N, 32 + 3*C)
        feature_names: list
    """

    bands = {
        "delta": (0.5, 4),
        "theta": (4, 8),
        "alpha": (8, 13),
        "beta":  (13, 30),
        "gamma": (30, 45),
    }

    N, C, T = X.shape
    features = []
    feature_names = []

    # ---------- FEATURE NAMES ----------
    for ch in range(C):
        prefix = f"ch{ch}"

        # spectral powers
        for band in bands:
            feature_names.append(f"{prefix}_{band}_power")
        for band in bands:
            feature_names.append(f"{prefix}_log_{band}_power")

        # ratios + complexity
        feature_names += [
            f"{prefix}_alpha_relative",
            f"{prefix}_alpha_beta_ratio",
            f"{prefix}_theta_alpha_ratio",
            f"{prefix}_beta_alpha_ratio",
            f"{prefix}_spectral_entropy",
            f"{prefix}_log_total_power",
        ]

        # time-domain stats (NEW)
        feature_names += [
            f"{prefix}_time_mean",
            f"{prefix}_time_std",
            f"{prefix}_time_var",
        ]

    # ---------- FEATURE EXTRACTION ----------
    for i in range(N):
        feat_vec = []

        for ch in range(C):
            signal = X[i, ch]

            # ----- PSD -----
            f, Pxx = welch(signal, fs=fs, nperseg=fs // 2)

            band_powers = {}
            total_power = 0.0

            for band, (fmin, fmax) in bands.items():
                mask = (f >= fmin) & (f <= fmax)
                power = np.trapezoid(Pxx[mask], f[mask])
                band_powers[band] = power
                total_power += power

            # ----- raw band powers -----
            for band in bands:
                feat_vec.append(band_powers[band])

            # ----- log band powers -----
            for band in bands:
                feat_vec.append(np.log(band_powers[band] + 1e-8))

            # ----- ratios -----
            alpha_rel   = band_powers["alpha"] / (total_power + 1e-8)
            alpha_beta  = band_powers["alpha"] / (band_powers["beta"] + 1e-8)
            theta_alpha = band_powers["theta"] / (band_powers["alpha"] + 1e-8)
            beta_alpha  = band_powers["beta"]  / (band_powers["alpha"] + 1e-8)

            # ----- spectral entropy -----
            Pxx_norm = Pxx / (Pxx.sum() + 1e-8)
            spec_entropy = -np.sum(Pxx_norm * np.log(Pxx_norm + 1e-8))

            log_total_power = np.log(total_power + 1e-8)

            feat_vec += [
                alpha_rel,
                alpha_beta,
                theta_alpha,
                beta_alpha,
                spec_entropy,
                log_total_power,
            ]

            # ----- time-domain stats (NEW) -----
            feat_vec += [
                signal.mean(),
                signal.std(),
                signal.var(),
            ]

        features.append(feat_vec)

    return np.array(features), feature_names

features, feature_names = extract_band_features_per_channel(X_clean, fs=250)
print("Extracted features shape:", features.shape)

np.savez(
    "datasets/clean_eye_data.npz",
    X=X_clean,
    y=y_clean,
    X_feat=features
)